In [ ]:
import os
from pathlib import Path
import shutil

import teehr

teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

In [ ]:
# create local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    create_dir=True
)

ev.enable_logging()

### Populate evaluation

In [ ]:
# Default configuration (public TEEHR warehouse)
your_api_key = 'your_key_here'  # Replace with your actual API key

ev.download.configure(
    api_key=your_api_key
)

In [ ]:
## ATTRIBUTES
ev.download.attributes(load=True)

In [ ]:
## LOCATIONS 

locations_subset = [
    "usgs-10011500",
    "usgs-10128500",
    "usgs-09081600",
]

ev.download.locations(
    prefix='usgs',
    ids=locations_subset,
    load=True,
)

In [ ]:
## LOCATION_ATTRIBUTES

ev.download.location_attributes(
    location_id=locations_subset,
    load=True
)

In [ ]:
## LOCATION_CROSSWALKS

import pandas as pd

crosswalk_data = {
    "primary_location_id": ["usgs-10011500", "usgs-10128500", "usgs-09081600"],
    "secondary_location_id":["hefs-BERU1HAF", "hefs-OAWU1HAF", "hefs-RCYC2HAF"]
}

crosswalk_df = pd.DataFrame(crosswalk_data)

ev.location_crosswalks.load_dataframe(crosswalk_df)

In [ ]:
## UNITS (remote download returns error, manually defined)

from teehr import Unit
unit = Unit(
    name="m^3/s",
    long_name="Cubic meters per second"
)
ev.units.add(unit)

In [ ]:
## VARIABLES

ev.download.variables(
    name="streamflow_hourly_inst",
    load=True
)

In [ ]:
## CONFIGURATIONS

ev.download.configurations(
    name='usgs_observations',
    load=True
)

from teehr import Configuration

configuration = Configuration(
    name="hefs_streamflow_forecast",
    timeseries_type="secondary",
    description="HEFS Streamflow Forecast",
)

ev.configurations.add(configuration)

In [ ]:
## PRIMARY_TIMESERIES

# the below is failing due to page_size bug
'''
for location in locations_subset:
    print(f"starting {location}...")
    ev.download.primary_timeseries(
        primary_location_id=location,
        configuration_name='usgs_observations',
        variable_name='streamflow_hourly_inst',
        start_datetime="1999-10-01 00:00:00",
        end_datetime="2005-09-30 23:00:00",
        load=True,
        #page_size = 5000000,
        #timeout=200,
    )
'''

# the below is failing due to page_size bug
'''
ev.download.primary_timeseries(
        primary_location_id=locations_subset,
        configuration_name='usgs_observations',
        variable_name='streamflow_hourly_inst',
        start_datetime="1999-10-01 00:00:00",
        end_datetime="2005-09-30 23:00:00",
        load=True,
        #page_size = 5000000,
        #timeout=200,
    )
'''

# legacy fetching for now
tbl = ev.table('primary_timeseries')
schema = tbl.schema_func().to_structtype()
options = {
    "header": "true",
    "ignoreMissingFiles": "true"
}
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e3_usgs_hourly_streamflow/dataset/primary_timeseries/"
sdf = ev.spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 
sdf = sdf.filter(sdf.location_id.isin(locations_subset))
ev._write.to_warehouse(table_name="primary_timeseries", source_data=sdf)


In [ ]:
## SECONDARY_TIMESERIES

inputs_dir = "inputs_dir/inputs_v2_trimmed/"
# sample: inputs/200001011200_BERU1HAF_MEFP_QINE.xml

ev.secondary_timeseries.load_fews_xml(
    in_path=inputs_dir,
    pattern="*.xml",
    constant_field_values={
        "unit_name":"m^3/s",
        "variable_name":"streamflow_hourly_inst",
        "configuration_name":"hefs_streamflow_forecast",
    },
    location_id_prefix="hefs"
)

### Kill Spark

In [ ]:
ev.spark.stop()